Biometric Technologies and Behavioural Security
# **<center>Tutorial 9 - Crowd Analysis and Blob detection</center>**
### **<center>Part 2: Face Mask Detector</center>**

##Step 1: Download Face Mask dataset and face detector files

This dataset consists of 1,376 images belonging to two classes:

* **with_mask**: 690 images
* **without_mask**: 686 images

In order to increase the generalizability of the model, the dataset is composed also by **augmented** images, that are the results of applying data augmentation to the original dataset.

In [ ]:
from IPython.display import clear_output
from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

downloaded = drive.CreateFile({'id': '1eWw_sGrm1huHWWIjUHq8W0terltF4Gh_'})
downloaded.GetContentFile('dataset.zip')
!unzip dataset
clear_output()
print('Done! Press the Refresh button if files are not visible.')

### Step 1.1 Prepare data and labels

In [ ]:
import os,cv2
import numpy as np
from IPython.display import HTML, display
import time

#Function for progress bar.
def progress(value, max=100):
    return HTML("""<progress value='{value}' max='{max}', style='width: 100%'>
            {value}
        </progress>""".format(value=value, max=max))

path = "/content/dataset/"
images = os.listdir(path)
print('Extracting faces...')
out = display(progress(0, len(images)*2), display_id=True)
kk=0
t = time.time()
data = []
labels = []

for i in images:
    path = path + i
    images = os.listdir(path)
    for j in images:
        img = cv2.imread(path+"/"+j)
        try:
          resized = cv2.resize(img, (224,224), interpolation = cv2.INTER_AREA)
          data.append(resized)
          if os.path.basename(os.path.normpath(path)) == "with_mask":
              labels.append(0)
          elif os.path.basename(os.path.normpath(path)) == "without_mask":
              labels.append(1)
        except:
          pass
        kk=kk+1
        out.update(progress(kk, len(images)*2))
    path = "/content/dataset/"

elapsed = 'Process completed! Elapsed time: %.2f s' %(time.time() - t)
print(elapsed)

Extracting faces...


Process completed! Elapsed time: 12.11 s


### Step 1.2: Data normalization

In [ ]:
X = np.array(data)
Y = np.array(labels)

from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(X,Y,test_size = 0.1)
#train and test norm
x_train = x_train.astype('float32')
x_test = x_test.astype('float32')
x_train /= 255
x_test /= 255
x_train = (x_train - 0.5) * 2
x_test = (x_test - 0.5) * 2

print('x_train shape:', x_train.shape)
print(x_train.shape[0], 'train samples')
print(x_test.shape[0], 'test samples')

x_train shape: (1238, 224, 224, 3)
1238 train samples
138 test samples


### Step 2: Define and Train a Neural Network

Build your own NN, with at least 3 hidden layers. Feel free to try different combinations of epochs/output numbers/etc.

Note: call the variable **model**

In [ ]:
import keras
from keras.layers import Conv2D, MaxPooling2D
from keras import backend as K
from keras.layers import Dense, Dropout, Flatten, Activation
from keras.models import Sequential, Model, load_model, model_from_yaml

num_classes=2
# input image dimensions
img_rows, img_cols = 224, 224

Using TensorFlow backend.


In [ ]:
input_shape = (224, 224, 3)

model = Sequential()
model.add(Conv2D(32, (3, 3), input_shape=input_shape))
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Conv2D(32, (3, 3)))
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Conv2D(64, (3, 3)))
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Flatten())
model.add(Dense(64))
model.add(Activation('relu'))
model.add(Dropout(0.5))
model.add(Dense(1))
model.add(Activation('sigmoid'))

model.compile(loss=keras.losses.binary_crossentropy,
              optimizer='Adam',
              metrics=['accuracy'])

model.fit(x_train, y_train,
          batch_size=20,
          epochs=25,
          verbose=1,
          validation_data=(x_test, y_test))

### Step 3: Test the model on a video stream

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode, b64encode
from PIL import Image
import io
import time


def VideoCapture():
  js = Javascript('''
    async function create(){
      div = document.createElement('div');
      document.body.appendChild(div);

      video = document.createElement('video');
      video.setAttribute('playsinline', '');

      div.appendChild(video);

      stream = await navigator.mediaDevices.getUserMedia({video: {facingMode: "environment"}});
      video.srcObject = stream;

      await video.play();

      canvas =  document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);

      div_out = document.createElement('div');
      document.body.appendChild(div_out);
      img = document.createElement('img');
      div_out.appendChild(img);

    }

    async function capture(){
        return await new Promise(function(resolve, reject){
            pendingResolve = resolve;
            canvas.getContext('2d').drawImage(video, 0, 0);
            result = canvas.toDataURL('image/jpeg', 0.8);
            pendingResolve(result);
        })
    }

    function showimg(imgb64){
        img.src = "data:image/jpg;base64," + imgb64;
    }

  ''')
  display(js)

def byte2image(byte):
  jpeg = b64decode(byte.split(',')[1])
  im = Image.open(io.BytesIO(jpeg))
  return np.array(im)

def image2byte(image):
  image = Image.fromarray(image)
  buffer = io.BytesIO()
  image.save(buffer, 'jpeg')
  buffer.seek(0)
  x = b64encode(buffer.read()).decode('utf-8')
  return x

In [ ]:



from google.colab.patches import cv2_imshow
import dlib
from skimage import feature

Dict = {0: 'Mask' ,1: 'No Mask'}
VideoCapture()
eval_js('create()')

print("[1] Loading face detector model...")
prototxtPath = "/content/face_detector/deploy.prototxt"
weightsPath =  "/content/face_detector/res10_300x300_ssd_iter_140000.caffemodel"
net = cv2.dnn.readNet(prototxtPath, weightsPath)


time.sleep(2.0)
while True:
    # grab the frame from the threaded video stream and resize it
    # to have a maximum width of 400 pixels

  byte = eval_js('capture()')
  image = byte2image(byte)


  (h, w) = image.shape[:2]
  # construct a blob from the image
  blob = cv2.dnn.blobFromImage(image, 1.0, (300, 300),(104.0, 177.0, 123.0))
  # pass the blob through the network and obtain the face detections
  net.setInput(blob)
  detections = net.forward()

  # loop over the detections
  for i in range(0, detections.shape[2]):
    confidence = detections[0, 0, i, 2]
    if confidence > 0.5:
      box = detections[0, 0, i, 3:7] * np.array([w, h, w, h])
      (startX, startY, endX, endY) = box.astype("int")
      (startX, startY) = (max(0, startX), max(0, startY))
      (endX, endY) = (min(w - 1, endX), min(h - 1, endY))
      face = image[startY:endY, startX:endX]
      face = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)
      face = cv2.resize(face, (224, 224))

      #prediction
      n = int(model.predict_classes(face.reshape(1,224,224,3)))
      if n==1:
        withoutMask = 1
        mask = 0
      else:
        mask=1
        withoutMask = 0

      label = "Mask" if mask > withoutMask else "No Mask"
      color = (0, 255, 0) if label == "Mask" else (0, 0, 255)
      cv2.putText(image, label, (startX, startY - 10),cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 2)
      cv2.rectangle(image, (startX, startY), (endX, endY), color, 2)

      eval_js('showimg("{}")'.format(image2byte(image)))

MessageError: ignored